In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler

In [2]:
# load eda outputs
df_train = pd.read_parquet("/work/artifacts/df_train_cleaned.parquet")
df_test  = pd.read_parquet("/work/artifacts/df_test_cleaned.parquet")
outlet_tr = pd.read_csv("/work/artifacts/outlet_tr.csv")['Outlet_Identifier'].values
sample_sub = pd.read_csv("/work/sample_submission_8RXa3c6.csv")
# Raw files needed only for high-cardinality TE columns (Item_Identifier etc.)
orig_train = pd.read_csv("/work/train_v9rqX0R.csv")
orig_test  = pd.read_csv("/work/test_AbJTz2l.csv")

In [3]:
print(df_train.shape, df_test.shape)

(8523, 13) (5681, 12)


In [4]:
# setup constants
N_MRP_BINS = 20

In [5]:
#mrp bins
_, mrp_edges = pd.qcut(df_train['Item_MRP'], N_MRP_BINS, retbins=True, duplicates='drop')
mrp_edges[0]  = -np.inf
mrp_edges[-1] =  np.inf

In [6]:
# feature engineering
for df in (df_train, df_test):
    df['Item_Category'] = df['Item_Identifier'].str[:2]
    df['Outlet_Age'] = 2013 - df['Outlet_Establishment_Year']
    df['Item_Visibility_Is_Zero'] = (df['Item_Visibility'] == 0).astype(int)
    df['MRP_bin'] = pd.cut(df['Item_MRP'], bins=mrp_edges, labels=False, include_lowest=True)
    
    df['Outlet_Type_Item_Type'] = df['Outlet_Type'] + '_' + df['Item_Type']
    
    df['MRP_x_Vis'] = df['Item_MRP'] * df['Item_Visibility'] 
    df['MRP_x_Age'] = df['Item_MRP'] * df['Outlet_Age']
    df['Price_per_Unit_Weight'] = df['Item_MRP'] / (df['Item_Weight'])
    
    df['log_MRP'] = np.log1p(df['Item_MRP'])

# Replace zero visibility
vis_mean_by_type = (df_train.loc[df_train['Item_Visibility'] > 0]
                            .groupby('Item_Type')['Item_Visibility'].mean())

for df in (df_train, df_test):
    mask = df['Item_Visibility'] == 0
    df.loc[mask, 'Item_Visibility'] = df.loc[mask, 'Item_Type'].map(vis_mean_by_type)

print(f"[FE complete] Train shape: {df_train.shape}")

[FE complete] Train shape: (8523, 22)


In [7]:
# 1. One-Hot Encode safely (combining train+test prevents missing columns)
one_hot_cols = ['Item_Fat_Content', 'Item_Type', 'Outlet_Size',
                'Outlet_Location_Type', 'Outlet_Type', 'Item_Category']

df_train = pd.get_dummies(df_train, columns=one_hot_cols, dtype=int)
df_test  = pd.get_dummies(df_test, columns=one_hot_cols, dtype=int)

# 2. Ensure Test set has exact same columns as Train set
df_test = df_test.reindex(columns=df_train.columns, fill_value=0)

# 3. Define X and y by dropping columns Ridge doesn't need
drop_cols = ['Item_Outlet_Sales', 'Item_Identifier', 'Outlet_Identifier',
             'Outlet_Establishment_Year', 'Outlet_Type_Item_Type']

X = df_train.drop(columns=drop_cols)
y = df_train['Item_Outlet_Sales']
X_test = df_test.drop(columns=drop_cols) # y is not in test, so no need to check if it exists

# 4. Final Safety Checks
# 4. Final Safety Checks
print("Columns match:", list(X.columns) == list(X_test.columns))
print("Missing values in Train:", X.isna().sum().sum())
print("Missing values in Test:", X_test.isna().sum().sum())
print("X:", X.shape, "| X_test:", X_test.shape, "| y:", y.shape)
print("X:", X.shape, "| X_test:", X_test.shape, "| y:", y.shape)

Columns match: True
Missing values in Train: 0
Missing values in Test: 0
X: (8523, 42) | X_test: (5681, 42) | y: (8523,)
X: (8523, 42) | X_test: (5681, 42) | y: (8523,)


In [8]:
#Setup and Target Encoding functions
RNG = 42
N_SPLITS = 10
SMOOTHING = 25

In [9]:
def rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))

def te_smoothed_mean(train_df, key_col, target_col, smoothing=SMOOTHING):
    gm  = train_df[target_col].mean()
    agg = train_df.groupby(key_col)[target_col].agg(['mean', 'count'])
    sm  = (agg['mean'] * agg['count'] + gm * smoothing) / (agg['count'] + smoothing)
    return sm, gm

def te_oof(train_df, key_col, target_col, n_inner=5, smoothing=SMOOTHING, seed=42):
    gm  = train_df[target_col].mean()
    out = np.full(len(train_df), gm, dtype=np.float64)
    rng = np.random.RandomState(seed)
    idx = rng.permutation(len(train_df))
    for f_idx in np.array_split(idx, n_inner):
        mask = np.ones(len(train_df), dtype=bool)
        mask[f_idx] = False
        sm, gm_fit = te_smoothed_mean(train_df.iloc[mask], key_col, target_col, smoothing)
        out[f_idx] = train_df.iloc[f_idx][key_col].map(sm).fillna(gm_fit).values
    return out

def te_apply(fit_df, apply_series, key_col, target_col, smoothing=SMOOTHING):
    sm, gm = te_smoothed_mean(fit_df, key_col, target_col, smoothing)
    return apply_series.map(sm).fillna(gm).values

In [10]:
# Stratification key: outlet x above/below median sales
median_y  = np.median(y.values)
strat_key = np.array([f"{o}_{int(s > median_y)}" for o, s in zip(outlet_tr, y.values)])

TE_COLS = {
    'Item_Identifier_te':       'Item_Identifier',
    'Outlet_Identifier_te':     'Outlet_Identifier',
    'Outlet_Type_Item_Type_te': 'Outlet_Type_Item_Type',
}

In [11]:
X_clean = X.copy()
X_test_clean = X_test.copy()

# Then your re-attach block:
X_clean = X.copy()
X_test_clean = X_test.copy()
X_clean['Item_Identifier'] = orig_train['Item_Identifier'].values
X_clean['Outlet_Identifier'] = orig_train['Outlet_Identifier'].values
X_clean['Outlet_Type_Item_Type'] = (orig_train['Outlet_Type'] + '_' + orig_train['Item_Type']).values

X_test_clean['Item_Identifier'] = orig_test['Item_Identifier'].values
X_test_clean['Outlet_Identifier'] = orig_test['Outlet_Identifier'].values
X_test_clean['Outlet_Type_Item_Type'] = (orig_test['Outlet_Type'] + '_' + orig_test['Item_Type']).values

In [14]:
# CV LOOP
kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RNG)
median_y  = np.median(y.values)
strat_key = np.array([f"{o}_{int(s > median_y)}" for o, s in zip(outlet_tr, y.values)])

w_train = np.ones(len(X_clean), dtype=np.float32)

oof_ridge  = np.zeros(len(X_clean))
pred_ridge = np.zeros(len(X_test_clean))

for fold, (tr, va) in enumerate(kf.split(X_clean, strat_key), 1):
    X_tr_df = X_clean.iloc[tr].copy()
    X_va_df = X_clean.iloc[va].copy()
    X_te_df = X_test_clean.copy()

    y_tr = y.iloc[tr].values
    y_va = y.iloc[va].values
    w_tr = w_train[tr]

    # Target encodings (fit on fold-train only, OOF internally) 
    for te_name, raw_col in TE_COLS.items():
        tmp = X_tr_df[[raw_col]].copy()
        tmp['__y__'] = y_tr
        X_tr_df[te_name] = te_oof(tmp, raw_col, '__y__', n_inner=5, seed=RNG + fold)
        X_va_df[te_name] = te_apply(tmp, X_va_df[raw_col], raw_col, '__y__')
        X_te_df[te_name] = te_apply(tmp, X_te_df[raw_col], raw_col, '__y__')

    drop_raw = list(TE_COLS.values())
    X_tr_f = X_tr_df.drop(columns=drop_raw).astype(np.float32)
    X_va_f = X_va_df.drop(columns=drop_raw).astype(np.float32)
    X_te_f = X_te_df.drop(columns=drop_raw).astype(np.float32)

    # Ridge 
    ss = StandardScaler()
    X_tr_scaled = ss.fit_transform(X_tr_f.astype(np.float64))
    X_va_scaled = ss.transform(X_va_f.astype(np.float64))
    X_te_scaled = ss.transform(X_te_f.astype(np.float64))

    ridge = RidgeCV(alphas=np.logspace(-3, 3, 40))
    ridge.fit(X_tr_scaled, y_tr, sample_weight=w_tr)
    oof_ridge[va] = ridge.predict(X_va_scaled)
    pred_ridge   += ridge.predict(X_te_scaled) / kf.n_splits

    print(f"[fold {fold:02d}/{N_SPLITS}] ridge={rmse(y_va, oof_ridge[va]):.4f} "
          f"(alpha={ridge.alpha_:.4f})")
print(f"\nOverall Ridge OOF RMSE: {rmse(y, oof_ridge):.4f}")

[fold 01/10] ridge=1101.4758 (alpha=0.1425)
[fold 02/10] ridge=1093.9531 (alpha=0.2894)
[fold 03/10] ridge=1128.7734 (alpha=0.1425)
[fold 04/10] ridge=1180.8211 (alpha=0.1425)
[fold 05/10] ridge=1119.3017 (alpha=0.1425)
[fold 06/10] ridge=1167.2897 (alpha=0.2031)
[fold 07/10] ridge=1118.9180 (alpha=0.1425)
[fold 08/10] ridge=1094.5640 (alpha=0.2894)
[fold 09/10] ridge=1161.3675 (alpha=0.1425)
[fold 10/10] ridge=1089.0104 (alpha=0.1425)

Overall Ridge OOF RMSE: 1125.9874


In [16]:
#SUBMISSION
final_pred = np.clip(pred_ridge, 0, None)
submission = sample_sub.copy()
submission['Item_Outlet_Sales'] = final_pred
submission.to_csv('submission.csv', index=False)
print("Wrote submission.csv:", submission.shape)

Wrote submission.csv: (5681, 3)


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=6d372df8-eee0-413a-89a1-c6a537fcecfa' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>